# Module 3 — The Tool Layer

**Course:** Build Your First AI Agent from Scratch  
**Community:** [Autonomous MSP](https://skool.com/autonomous-msp-2162)

**What you build in this module:**
- A three-tier safety model (READ / WRITE / ADMIN)
- A `BaseTool` class every tool inherits from
- A `ToolRegistry` that generates LLM-readable descriptions
- `ShowCommandTool` — SSH show commands with a DEMO_MODE flag
- `ConfigDiffTool` — shows exactly what will change before any write
- `execute_with_approval()` — the safety gate that logs everything

**Key rule: the LLM never executes code. It only requests. You decide. You execute.**

## Step 1 — Install Dependencies

In [ ]:
# netmiko is for real SSH sessions — you won't use it in DEMO_MODE
# but the import is inside the tool so this only matters when you go live
!pip install netmiko -q

## Step 2 — Three Access Levels

Think of these like privilege levels on a Cisco router.

Every tool you build belongs to exactly one of three levels:

- **READ** — show commands, lookups, pings. Runs automatically. Cannot change anything.
- **WRITE** — config changes, interface bounces. Agent proposes, you type `y` to approve.
- **ADMIN** — high-impact operations. You must type `YES I CONFIRM` exactly.

You are running this agent against client infrastructure. The access level is not a suggestion.

In [ ]:
READ  = "read"    # privilege 1 — show commands only, auto-execute
WRITE = "write"   # privilege 7 — config changes, needs y/n approval
ADMIN = "admin"   # privilege 15 — destructive ops, needs YES I CONFIRM


class ToolResult:
    """
    Every tool returns this. Always the same three fields:
      success — True or False
      data    — what came back, e.g. {"raw_output": "..."}
      error   — error message if success is False, empty string otherwise
    """
    def __init__(self, success: bool, data: dict, error: str = ""):
        self.success = success
        self.data    = data
        self.error   = error


result = ToolResult(success=True, data={"output": "neighbor FULL"})
print(f"Access levels: {READ}, {WRITE}, {ADMIN}")
print(f"ToolResult: success={result.success}, data={result.data}")

## Step 3 — BaseTool: The Template Every Tool Follows

Think of `BaseTool` like a standard interface config template on a Cisco router.

The template defines what every interface *must* have: description, IP, OSPF settings.
You never activate the template itself — you apply it to a real interface and fill in the values.

`BaseTool` works the same way. It defines what every tool *must* declare:
- `name` — what the LLM types to call this tool
- `description` — what it does (the LLM reads this to decide which tool to use)
- `category` — READ, WRITE, or ADMIN
- `parameters` — what inputs it needs (the LLM reads this to know what to pass)
- `execute()` — the actual work

You never use `BaseTool` directly. You build a specific tool that inherits from it.
If you forget to write `execute()`, you get a clear error instead of silent failure.

In [ ]:
class BaseTool:
    """
    Template that every tool must follow.
    Inherit from this, fill in the four fields, implement execute().
    """
    name        = ""     # e.g. "run_show_command"
    description = ""     # e.g. "Execute a show command via SSH"
    category    = READ   # READ / WRITE / ADMIN
    parameters  = {}     # e.g. {"device": "hostname or IP"}

    def execute(self, **kwargs) -> ToolResult:
        # Subclasses must override this.
        # If they forget, they get a clear error — not silent failure.
        raise NotImplementedError(
            f"Tool '{self.name}' has no execute() method. "
            f"You must implement it before registering this tool."
        )


template = BaseTool()
template.name = "test"
try:
    template.execute()
except NotImplementedError as e:
    print(f"Good — BaseTool enforces the contract: {e}")

## Step 4 — ToolRegistry: The Agent's Toolbox

The registry solves two problems at once.

**Problem 1:** The LLM produces text like `run_show_command`. Your Python needs to find
the actual tool object that matches that name. Without a registry you need an `if/elif`
chain — one branch per tool. That breaks every time you add a tool.

**Problem 2:** The LLM needs to know what tools exist and what they do.
Without a registry you manually paste descriptions into the system prompt and
remember to update them whenever something changes.

The registry fixes both: register a tool once, and it is automatically findable
by name AND automatically included in the LLM system prompt.

In [ ]:
class ToolRegistry:
    """
    The agent's toolbox. Does two jobs:
    1. Stores tools so the agent can find them by name at runtime.
    2. Generates the text block injected into the LLM system prompt.
    """

    def __init__(self):
        self._tools = {}   # { tool_name: tool_object }

    def register(self, tool: BaseTool):
        """Add a tool to the toolbox."""
        self._tools[tool.name] = tool
        print(f"Registered: {tool.name} [{tool.category}]")

    def get(self, name: str):
        """Look up a tool by name. Returns None if not found."""
        return self._tools.get(name)

    def describe_all(self) -> str:
        """Generate the tool list for the LLM system prompt."""
        lines = []
        for tool in self._tools.values():
            lines.append(
                f"- {tool.name} [{tool.category}]: "
                f"{tool.description} | params: {tool.parameters}"
            )
        return "\n".join(lines)


registry = ToolRegistry()
print("Registry created.")

## Step 5 — ShowCommandTool (READ)

Your first real tool. Runs `show` commands on network devices via SSH.

**`DEMO_MODE = True`** — returns hardcoded output, no SSH or lab needed.  
**`DEMO_MODE = False`** — SSHs into a real device using Netmiko.

Flip one flag. Nothing else changes.

In [ ]:
DEMO_MODE = True  # flip to False when you have real lab devices

DEMO_DATA = {
    ("core-sw-01", "show ip ospf neighbor"): (
        "Neighbor ID     Pri   State      Dead Time   Interface\n"
        "10.0.0.2         1   FULL/DR    00:00:38    Gi0/1\n"
        "10.0.0.3         1   INIT       00:00:32    Gi0/1"
    ),
    ("core-sw-01", "show interfaces Gi0/1"): (
        "GigabitEthernet0/1 is up, line protocol is up\n"
        "  MTU 1500 bytes, BW 1000000 Kbit/sec"
    ),
    ("edge-rtr-01", "show ip ospf interface Gi0/1"): (
        "GigabitEthernet0/1 is up\n"
        "  OSPF enabled, Network type BROADCAST\n"
        "  Area 0.0.0.0, Cost: 1\n"
        "  Hello 10, Dead 40, Wait 40, Retransmit 5"
    ),
}


class ShowCommandTool(BaseTool):
    name        = "run_show_command"
    description = "Execute a show command on a network device via SSH"
    category    = READ
    parameters  = {
        "device":  "hostname or IP of the target device",
        "command": "the show command to run (e.g. 'show ip ospf neighbor')",
    }

    def execute(self, device: str, command: str) -> ToolResult:
        if DEMO_MODE:
            output = DEMO_DATA.get((device, command))
            if output is None:
                return ToolResult(
                    success=False, data={},
                    error=f"No demo data for device='{device}' command='{command}'"
                )
            return ToolResult(success=True, data={"raw_output": output})

        from netmiko import ConnectHandler
        conn = ConnectHandler(
            device_type="cisco_ios", host=device,
            username="netadmin", password="LOAD_FROM_ENV",
        )
        output = conn.send_command(command)
        conn.disconnect()
        return ToolResult(success=True, data={"raw_output": output})


show_tool = ShowCommandTool()
registry.register(show_tool)

print("\nTest — show ip ospf neighbor on core-sw-01:")
result = show_tool.execute("core-sw-01", "show ip ospf neighbor")
print(f"Success: {result.success}")
print(result.data["raw_output"])

## Step 6 — ConfigDiffTool (READ)

Before the agent proposes any config change, it shows exactly what will change — like a `git diff`.

Lines starting with `-` are removed. Lines starting with `+` are added.
The engineer sees this *before* the approval prompt for the WRITE operation.

Still READ — it only calculates the diff, it does not apply anything.

In [ ]:
import difflib

DEMO_DATA[("core-sw-01", "show running-config")] = (
    "interface GigabitEthernet0/1\n"
    " ip ospf 1 area 0\n"
    " ip ospf hello-interval 10\n"
    " no shutdown"
)


class ConfigDiffTool(BaseTool):
    name        = "show_config_diff"
    description = (
        "Compare current running config against a proposed config block. "
        "Always run this before proposing any configuration change."
    )
    category    = READ
    parameters  = {
        "device":          "hostname or IP",
        "proposed_config": "the config lines you intend to apply",
    }

    def execute(self, device: str, proposed_config: str) -> ToolResult:
        if DEMO_MODE:
            current = DEMO_DATA.get(
                (device, "show running-config"), "! no demo config available"
            )
        else:
            from netmiko import ConnectHandler
            conn = ConnectHandler(
                device_type="cisco_ios", host=device,
                username="netadmin", password="LOAD_FROM_ENV"
            )
            current = conn.send_command("show running-config")
            conn.disconnect()

        diff_lines = difflib.unified_diff(
            current.splitlines(), proposed_config.splitlines(),
            fromfile="running-config", tofile="proposed-config", lineterm=""
        )
        return ToolResult(success=True, data={"diff": "\n".join(diff_lines)})


diff_tool = ConfigDiffTool()
registry.register(diff_tool)

print("Test — config diff (area 0 → area 1):")
proposed = (
    "interface GigabitEthernet0/1\n"
    " ip ospf 1 area 1\n"
    " ip ospf hello-interval 10\n"
    " no shutdown"
)
diff_result = diff_tool.execute("core-sw-01", proposed)
print(diff_result.data["diff"])

## Step 7 — The Safety Gate: execute_with_approval()

**All tool calls go through here. No exceptions.**

- READ  → runs immediately, logs it
- WRITE → shows what will change, asks `y/n`, logs the decision
- ADMIN → requires typing `YES I CONFIRM` exactly, logs the decision

`audit.jsonl` writes one JSON line per action — approved or rejected.
This is your proof trail for clients and auditors.

In [ ]:
import json
import datetime

AUDIT_LOG_PATH = "audit.jsonl"


def _log_action(tool_name: str, params: dict, approved: bool, operator: str):
    entry = {
        "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
        "tool": tool_name, "params": params,
        "approved": approved, "operator": operator,
    }
    with open(AUDIT_LOG_PATH, "a") as f:
        f.write(json.dumps(entry) + "\n")


def execute_with_approval(tool: BaseTool, params: dict, operator: str = "unknown") -> ToolResult:
    """The safety gate. Every tool call goes through here."""

    if tool.category == READ:
        result = tool.execute(**params)
        _log_action(tool.name, params, approved=True, operator=operator)
        return result

    if tool.category == WRITE:
        print(f"\n{'='*55}")
        print("  WRITE OPERATION REQUESTED")
        print(f"{'='*55}")
        print(f"  Tool  : {tool.name}")
        print(f"  Params: {json.dumps(params, indent=4)}")
        if "diff" in params:
            print(f"\n  Config diff:\n{params['diff']}")
        print(f"{'='*55}")
        answer   = input("  Approve? (y/n): ").strip().lower()
        approved = (answer == "y")
        _log_action(tool.name, params, approved=approved, operator=operator)
        if not approved:
            return ToolResult(success=False, data={}, error="Operator rejected.")
        return tool.execute(**params)

    if tool.category == ADMIN:
        print(f"\n{'!'*55}")
        print("  ADMIN OPERATION — HIGH IMPACT")
        print(f"  Tool  : {tool.name}")
        print("  Type 'YES I CONFIRM' to proceed:")
        answer   = input("  > ").strip()
        approved = (answer == "YES I CONFIRM")
        _log_action(tool.name, params, approved=approved, operator=operator)
        if not approved:
            return ToolResult(success=False, data={}, error="Admin confirmation not received.")
        return tool.execute(**params)

    return ToolResult(success=False, data={}, error="Unknown tool category.")


print("Safety gate ready.")

## Step 8 — Test the Full Flow

This cell demonstrates the complete flow:
1. Agent requests a READ tool → runs automatically
2. Check the audit log to confirm it was recorded

In [ ]:
print("=== Test: READ tool (auto-executes) ===")
result = execute_with_approval(
    tool=registry.get("run_show_command"),
    params={"device": "core-sw-01", "command": "show ip ospf neighbor"},
    operator="ed.dulharu"
)
print(f"Success: {result.success}")
print(f"Output:\n{result.data.get('raw_output', result.error)}")

print("\n=== Audit log (last entry) ===")
with open(AUDIT_LOG_PATH) as f:
    lines = f.readlines()
    print(json.dumps(json.loads(lines[-1]), indent=2))

## Step 9 — See What the LLM Sees

The registry's `describe_all()` output is what you inject into the system prompt. Run this cell to see exactly what the LLM reads when deciding which tool to call.

In [ ]:
print("=== What the LLM sees in the system prompt ===")
print(registry.describe_all())

## What's Next

You now have a complete, safe tool layer. In **Module 4** you add memory: ChromaDB for past incidents, so your agent can say "I've seen this before" instead of starting from scratch every time.

---

**Module Challenge:** Write one READ tool for your environment — a specific `show` command you run every week, or a query to your ticketing system. Post the tool name and one-line description in **#agent-builds**.